In [ ]:
import json
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader

import fluidbench
from fluidbench.cfdataset import CFDataset

In [ ]:
ID = "cylinder_Re41_interp"
dataset_path = fluidbench.download(ID)

In [ ]:
def load_split(split):
    folder = dataset_path / split
    cases = sorted((path for path in folder.glob("*.h5") if path.stem != "constants"), key=lambda path: int(path.stem))
    data = []
    for case in cases:
        with h5py.File(case, "r") as file:
            data.append((file["scalars"][:], file["fields"][:]))
    return data

def to_dataset(cases):
    # Each snapshot is paired with the next one: inputs are snapshot i, targets snapshot i + 1.
    scalars = np.concatenate([scalars[:-1] for scalars, fields in cases])
    inputs = np.concatenate([fields[:-1] for scalars, fields in cases])
    targets = np.concatenate([fields[1:] for scalars, fields in cases])
    return CFDataset(scalars, inputs, targets, np.arange(len(targets)))

In [ ]:
### Model definition
from models.minimal_models import BaselineZero as architecture

weights_path = Path("weights") / ID / f"{architecture.__name__}.pt"

model = architecture()
model.load_weights(weights_path)
model.to(model.device)

In [ ]:
# Metrics are calculated by fluidbench.run() from the benchmark test split.
submission_config = {
    "datasetId": ID,
    "id": "baseline-zero",
    "model": "Baseline Zero",
    "authors": "Pablo A.C.",
    "year": 2026,
    "representation": "Grid",
}

# Optional: training curves saved by 2.1.train
training_history = json.loads(weights_path.with_suffix(".json").read_text())

In [ ]:
submission = fluidbench.run(model, submission_config, training_history)